# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pip install -U datasets huggingface_hub

In [3]:
from huggingface_hub import login

login()

In [4]:
from datasets import load_dataset
from datasets import load_dataset

ds = load_dataset("FlyRank/internship-warehouse", "dim_clients")
print(ds)
print(ds["train"].column_names)

dim_clients.parquet: reconstructing file:   0%|          |  0.00B / 3.38kB            

dim_clients.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/104 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['client_hash_id', 'is_active', 'has_gsc_access', 'has_ga4_access', 'access_profile', 'client_created_date', 'client_updated_date', 'gsc_data_start', 'ga4_data_start'],
        num_rows: 104
    })
})
['client_hash_id', 'is_active', 'has_gsc_access', 'has_ga4_access', 'access_profile', 'client_created_date', 'client_updated_date', 'gsc_data_start', 'ga4_data_start']


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of Analysis + Time Window

For this data contract, **one row represents one client** in the FlyRank warehouse. Each record stores the client's access status and the availability of Google Search Console (GSC) and Google Analytics 4 (GA4) data.

The available time information is provided through the client lifecycle fields (`client_created_date` and `client_updated_date`) and the data availability fields (`gsc_data_start`). This dataset is a client dimension table rather than a daily performance table.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features:

    is_active
    has_gsc_access
    has_ga4_access

Label:

    access_profile

Context:

    client_created_date
    client_updated_date

Excluded:

    client_hash_id (hashed identifier, not useful for prediction)


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("FlyRank/internship-warehouse", "dim_clients")
df = pd.DataFrame(ds["train"])

# Convert 'client_created_date' and 'client_updated_date' to datetime, coercing errors
df["client_created_date"] = pd.to_datetime(df["client_created_date"], errors='coerce')
df["client_updated_date"] = pd.to_datetime(df["client_updated_date"], errors='coerce')

# Dataset shape
print("Shape:", df.shape)

# Check duplicate client IDs
print("\nDuplicate Client IDs:")
print(df["client_hash_id"].duplicated().sum())

# Missing values
print("\nMissing Values:")
print(df.isnull().sum())

# Access profile distribution
print("\nAccess Profile Distribution:")
print(df["access_profile"].value_counts())

# Active vs Inactive clients
print("\nActive Clients:")
print(df["is_active"].value_counts())

# Date range
print("\nClient Created Date Range:")
print(df["client_created_date"].min(), "to", df["client_created_date"].max())

Shape: (104, 9)

Duplicate Client IDs:
0

Missing Values:
client_hash_id          0
is_active              10
has_gsc_access         10
has_ga4_access         10
access_profile          0
client_created_date    10
client_updated_date    10
gsc_data_start         37
ga4_data_start         53
dtype: int64

Access Profile Distribution:
access_profile
gsc_and_ga4                             53
no_search_or_analytics_access           26
gsc_only                                14
source_only_missing_client_dimension    10
ga4_only                                 1
Name: count, dtype: int64

Active Clients:
is_active
True     74
False    20
Name: count, dtype: int64

Client Created Date Range:
2025-05-26 00:00:00 to 2026-06-29 00:00:00


## Verification

The verification confirms that:

- Each row represents one unique client.
- The dataset contains one record per client.
- Missing values are identified before analysis.
- The distribution of access profiles is measured.
- The available client creation dates define the observed time span of the dataset.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data Limits

This dataset contains client-level metadata only. It does not include search performance, page-level metrics, or user engagement information.

The dataset cannot explain how website content performs or predict future client behaviour. Some clients have missing GSC or GA4 information because access may not have been granted or the services were not connected.

Therefore, this analysis should be interpreted as **observational, measured, and decision-support**, rather than evidence of causal relationships or future outcomes.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.